# Phase 2 Task 1: Origin CV training

Reuse the final Phase 1 manifest keys and per-species five-fold assignment.
Protocol `best-val-origin-cv-v2`: each fold starts a new ImageNet-pretrained model, holds out about 20% of each species' training origins as inner validation, keeps the checkpoint with the best inner-val accuracy (warmup + cosine, early stopping patience 10, as Phase 1 Final-B) and evaluates the outer held-out fold once with that checkpoint.

Minimum scope: ConvNeXt-Tiny and MobileNetV3 × 5 folds. Recommended: all five backbones.


## 1. Setup

In [ ]:
from pathlib import Path
import sys

if Path('/content').exists():
    ROOT = Path('/content/woodbench')
else:
    ROOT = Path.cwd()

if not (ROOT / 'src').is_dir():
    ROOT = next(path for path in ROOT.parents if (path / 'src').is_dir())
sys.path.insert(0, str(ROOT))
TASK_DIR = ROOT / 'notebooks/phase 2/supervisor-feedback/task_01_fivefold_cv_training'
sys.path.insert(0, str(TASK_DIR))

import pandas as pd
from origin_cv import OriginKFoldDataset


## 2. Configuration

In [ ]:
# Training configuration
CONFIG = {
    'models': ['convnext_tiny', 'mobilenetv3_large', 'resnet50', 'efficientnet_b3', 'swin_tiny'],
    'n_folds': 5,
    'epochs': 50,  # Maximum per fold, matching Phase 1's configured budget
    'batch_size': None,  # Engine uses Phase 1 batch per model; set an integer to override
    'backbone_lr': None,  # REQUIRED: value chosen in step 1 (task_07_lr_selection)
    'head_lr': 1e-4,
    'warmup_epochs': 5,
    'early_stopping_patience': 10,
    'inner_val_fraction': 0.2,
    'input_variant': 'rgb',  # 'grayscale' or 'px16' for steps 3-4 (ConvNeXt, fold 1)
    'weight_decay': 1e-4,
    'label_smoothing': 0.1,
    'seed': 42,
    'num_workers': 4,
    'use_90rotation': False,  # Phase 1 main; True only for the 90-degree ablation
    'resume': False,  # True: keep matching completed folds, restart unfinished folds
}

DATA_DIR = ROOT / "data" / "task-B-group-split"
OUTPUT_DIR = ROOT / "data" / "experiments" / "Task-B-origin-cv-v2"
if CONFIG['input_variant'] != 'rgb':
    OUTPUT_DIR = OUTPUT_DIR.with_name(f"{OUTPUT_DIR.name}-{CONFIG['input_variant']}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")
print(f"\nOutput: {OUTPUT_DIR}")

## 3. Analyze Origin Distribution

In [ ]:
# Reuse the final Phase 1 manifest, including collision-renamed files.
full_dataset = OriginKFoldDataset(DATA_DIR, 'all')
df = pd.DataFrame(full_dataset.records)
df['full_path'] = df['path']
df['path'] = df['path'].map(lambda path: str(Path(path).relative_to(ROOT)))
df['filename'] = df['full_path'].map(lambda path: Path(path).name)
df['split'] = df['source_split']
cv_splits = full_dataset.get_splits_by_origin(
    CONFIG['n_folds'], seed=CONFIG['seed'], inner_val_fraction=CONFIG['inner_val_fraction']
)

print(f"Total samples: {len(df)}")
print(f"Unique Phase 1 origins: {df['origin_key'].nunique()}")
print(f"Species: {df['species'].nunique()}")
print("\n=== Origins per species ===")
print(df.groupby('species')['origin_key'].nunique().sort_values())
for split in cv_splits:
    print(f"Fold {split['fold'] + 1}: {len(split['inner_train_idx'])} train, "
          f"{len(split['inner_val_idx'])} val, {split['n_val']} held out")


## 4. Train with the shared CLI engine

Set `resume=True` to continue an interrupted experiment. A changed config, code or split requires a new output directory. An unfinished fold starts again at epoch 0; completed matching folds are retained. For steps 3-4 run only ConvNeXt fold 1 (`--fold 1`) with `input_variant` set.


In [ ]:
from train_kfold import main as train_cv

summaries = []
for backbone in CONFIG['models']:
    argv = ['--backbone', backbone, '--data_root', str(DATA_DIR),
            '--output_root', str(OUTPUT_DIR)]
    for key in ('n_folds', 'epochs', 'batch_size', 'backbone_lr', 'head_lr',
                'weight_decay', 'label_smoothing', 'seed', 'num_workers',
                'warmup_epochs', 'early_stopping_patience', 'inner_val_fraction',
                'input_variant'):
        if CONFIG[key] is not None:
            argv.extend(['--' + key, str(CONFIG[key])])
    if CONFIG['use_90rotation']:
        argv.append('--use_90rotation')
    if CONFIG['resume']:
        argv.append('--resume')
    summaries.append(train_cv(argv))


## 5. Check training completion

Run `evaluate_and_register_cv.ipynb` at this same level after all folds finish. It validates prediction artifacts and exports a new registry, per-class metrics, OOF predictions and fold summaries. Standard deviation describes fold variation and is not a confidence interval. Historical fixed-split comparisons need explicit baseline run IDs and their registry.


In [ ]:
pd.DataFrame([{'backbone': s['config']['backbone'], 'completed': s['n_completed'], 'n_folds': s['n_folds'], 'source_run_ids': s['source_run_ids']} for s in summaries])
